# Lambda & Serverless

## Lambda Basics

**AWS Lambda** is a serverless compute service. You write functions in Python, Node.js, Java, Go, or other languages. Lambda automatically scales from zero to thousands of concurrent executions.

You pay only for compute time consumed, measured in milliseconds. There's a generous free tier: 1 million requests and 400,000 GB-seconds per month.

## Lambda Triggers

Lambda functions are triggered by events from other AWS services:

**API Gateway** triggers Lambda when HTTP requests arrive.

**S3** triggers Lambda when objects are uploaded or deleted.

**DynamoDB Streams** trigger Lambda when data changes.

**SNS** triggers Lambda when messages are published.

**CloudWatch Events** trigger Lambda on a schedule (cron).

**SQS** triggers Lambda when messages arrive in a queue.

## API Gateway

**API Gateway** creates REST or WebSocket APIs that trigger Lambda functions. It handles authentication, rate limiting, and request/response transformation.

## Hands-On: Create Lambda Function

Create a simple Lambda function:

```bash
cat > lambda_function.py << 'EOF'
def lambda_handler(event, context):
    return {
        'statusCode': 200,
        'body': 'Hello from Lambda!'
    }
EOF
```

Package and deploy:

```bash
zip lambda.zip lambda_function.py

aws lambda create-function --function-name my-function \
  --runtime python3.11 --role arn:aws:iam::ACCOUNT:role/lambda-role \
  --handler lambda_function.lambda_handler --zip-file fileb://lambda.zip
```

Invoke the function:

```bash
aws lambda invoke --function-name my-function response.json
cat response.json
```

Create an API Gateway trigger:

```bash
aws apigateway create-rest-api --name my-api
aws apigateway create-resource --rest-api-id api-id --parent-id root-id \
  --path-part hello
aws apigateway put-method --rest-api-id api-id --resource-id resource-id \
  --http-method GET --authorization-type NONE
```

## Python Boto3 Example

In [ ]:
import boto3
import json

lambda_client = boto3.client('lambda')

# Create function
response = lambda_client.create_function(
    FunctionName='my-function',
    Runtime='python3.11',
    Role='arn:aws:iam::ACCOUNT:role/lambda-role',
    Handler='lambda_function.lambda_handler',
    Code={'ZipFile': b'function code here'}
)

# Invoke function
response = lambda_client.invoke(
    FunctionName='my-function',
    InvocationType='RequestResponse',
    Payload=json.dumps({'key': 'value'})
)

print(json.loads(response['Payload'].read()))

# Update function code
lambda_client.update_function_code(
    FunctionName='my-function',
    ZipFile=b'new function code'
)

## Serverless Application Model (SAM)

SAM is a framework for building serverless applications. Define your Lambda functions, API Gateway, and other resources in a YAML template:

```yaml
AWSTemplateFormatVersion: '2010-09-09'
Transform: AWS::Serverless-2016-10-31

Resources:
  MyFunction:
    Type: AWS::Serverless::Function
    Properties:
      FunctionName: my-function
      Runtime: python3.11
      Handler: lambda_function.lambda_handler
      CodeUri: .
      Events:
        ApiEvent:
          Type: Api
          Properties:
            RestApiId: !Ref MyApi
            Path: /hello
            Method: GET

  MyApi:
    Type: AWS::Serverless::Api
    Properties:
      StageName: prod
```

Deploy with SAM:

```bash
sam build
sam deploy --guided
```

## Terraform Example

```hcl
resource "aws_lambda_function" "my_function" {
  filename      = "lambda.zip"
  function_name = "my-function"
  role          = aws_iam_role.lambda_role.arn
  handler       = "lambda_function.lambda_handler"
  runtime       = "python3.11"
}

resource "aws_apigatewayv2_api" "my_api" {
  name          = "my-api"
  protocol_type = "HTTP"
}

resource "aws_apigatewayv2_integration" "lambda_integration" {
  api_id           = aws_apigatewayv2_api.my_api.id
  integration_type = "AWS_PROXY"
  integration_method = "POST"
  payload_format_version = "2.0"
  target           = aws_lambda_function.my_function.arn
}
```

## Quiz 1

<div class="quiz" data-correct="0">
  <p class="font-semibold mb-3">❓ What is AWS Lambda?</p>
  <div class="space-y-2">
    <label class="flex items-center gap-2 cursor-pointer">
      <input type="radio" name="q7384629" value="0">
      <span>A serverless compute service</span>
    </label>
    <label class="flex items-center gap-2 cursor-pointer">
      <input type="radio" name="q7384629" value="1">
      <span>A virtual machine service</span>
    </label>
    <label class="flex items-center gap-2 cursor-pointer">
      <input type="radio" name="q7384629" value="2">
      <span>A database service</span>
    </label>
    <label class="flex items-center gap-2 cursor-pointer">
      <input type="radio" name="q7384629" value="3">
      <span>A storage service</span>
    </label>
  </div>
  <button class="quiz-btn mt-3 px-4 py-2 bg-blue-600 text-white rounded text-sm font-medium hover:bg-blue-700">Check Answer</button>
  <p class="quiz-result text-sm mt-2 hidden"></p>
</div>

## Quiz 2

<div class="quiz" data-correct="2">
  <p class="font-semibold mb-3">❓ What is a Lambda trigger?</p>
  <div class="space-y-2">
    <label class="flex items-center gap-2 cursor-pointer">
      <input type="radio" name="q5738291" value="0">
      <span>A Lambda function parameter</span>
    </label>
    <label class="flex items-center gap-2 cursor-pointer">
      <input type="radio" name="q5738291" value="1">
      <span>A Lambda function version</span>
    </label>
    <label class="flex items-center gap-2 cursor-pointer">
      <input type="radio" name="q5738291" value="2">
      <span>An event from another AWS service that invokes Lambda</span>
    </label>
    <label class="flex items-center gap-2 cursor-pointer">
      <input type="radio" name="q5738291" value="3">
      <span>A Lambda function alias</span>
    </label>
  </div>
  <button class="quiz-btn mt-3 px-4 py-2 bg-blue-600 text-white rounded text-sm font-medium hover:bg-blue-700">Check Answer</button>
  <p class="quiz-result text-sm mt-2 hidden"></p>
</div>

## Quiz 3

<div class="quiz" data-correct="1">
  <p class="font-semibold mb-3">❓ What is API Gateway?</p>
  <div class="space-y-2">
    <label class="flex items-center gap-2 cursor-pointer">
      <input type="radio" name="q9384756" value="0">
      <span>A database service</span>
    </label>
    <label class="flex items-center gap-2 cursor-pointer">
      <input type="radio" name="q9384756" value="1">
      <span>A service that creates REST APIs and triggers Lambda</span>
    </label>
    <label class="flex items-center gap-2 cursor-pointer">
      <input type="radio" name="q9384756" value="2">
      <span>A networking service</span>
    </label>
    <label class="flex items-center gap-2 cursor-pointer">
      <input type="radio" name="q9384756" value="3">
      <span>A caching service</span>
    </label>
  </div>
  <button class="quiz-btn mt-3 px-4 py-2 bg-blue-600 text-white rounded text-sm font-medium hover:bg-blue-700">Check Answer</button>
  <p class="quiz-result text-sm mt-2 hidden"></p>
</div>

## Quiz 4

<div class="quiz" data-correct="0">
  <p class="font-semibold mb-3">❓ How is Lambda pricing calculated?</p>
  <div class="space-y-2">
    <label class="flex items-center gap-2 cursor-pointer">
      <input type="radio" name="q4729183" value="0">
      <span>Per request and compute time in milliseconds</span>
    </label>
    <label class="flex items-center gap-2 cursor-pointer">
      <input type="radio" name="q4729183" value="1">
      <span>Per hour of execution</span>
    </label>
    <label class="flex items-center gap-2 cursor-pointer">
      <input type="radio" name="q4729183" value="2">
      <span>Per GB of memory allocated</span>
    </label>
    <label class="flex items-center gap-2 cursor-pointer">
      <input type="radio" name="q4729183" value="3">
      <span>Fixed monthly fee</span>
    </label>
  </div>
  <button class="quiz-btn mt-3 px-4 py-2 bg-blue-600 text-white rounded text-sm font-medium hover:bg-blue-700">Check Answer</button>
  <p class="quiz-result text-sm mt-2 hidden"></p>
</div>

## Quiz 5

<div class="quiz" data-correct="1">
  <p class="font-semibold mb-3">❓ What is SAM?</p>
  <div class="space-y-2">
    <label class="flex items-center gap-2 cursor-pointer">
      <input type="radio" name="q8293847" value="0">
      <span>A Lambda function</span>
    </label>
    <label class="flex items-center gap-2 cursor-pointer">
      <input type="radio" name="q8293847" value="1">
      <span>A framework for building serverless applications</span>
    </label>
    <label class="flex items-center gap-2 cursor-pointer">
      <input type="radio" name="q8293847" value="2">
      <span>A database service</span>
    </label>
    <label class="flex items-center gap-2 cursor-pointer">
      <input type="radio" name="q8293847" value="3">
      <span>A monitoring service</span>
    </label>
  </div>
  <button class="quiz-btn mt-3 px-4 py-2 bg-blue-600 text-white rounded text-sm font-medium hover:bg-blue-700">Check Answer</button>
  <p class="quiz-result text-sm mt-2 hidden"></p>
</div>